In [4]:
import yfinance as yf
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN, # RNN 은닉층
    Dense 
)
from sklearn.metrics import r2_score

In [5]:
# 1. 삼성전자 최근 3년 주가 데이터 수집
df = yf.download(
    "005930.KS",
    period="3y",
    interval="1d",
    auto_adjust=True,
    progress=False
)

In [6]:
# 2. 날짜와 종가만 선택
df = df.reset_index()
df = df[["Date", "Close"]]
df.columns = ["날짜", "종가"]
df = df.reset_index(drop=True)

display(df.head(21))

,날짜,종가
0,2023-10-23,65024.507812
1,2023-10-24,65119.574219
2,2023-10-25,64644.246094
3,2023-10-26,63408.394531
4,2023-10-27,63978.796875
5,2023-10-30,63978.796875
6,2023-10-31,63598.535156
7,2023-11-01,65214.640625
8,2023-11-02,66260.359375
9,2023-11-03,66165.281250


In [7]:
# 3. 결측치 확인
display(df.isna().sum())

날짜    0
종가    0
dtype: int64

In [8]:
# 4. 종가를 NumPy 배열로 변환
data = df["종가"].to_numpy().reshape(-1, 1)
print('data.shape:', data.shape)

data.shape: (729, 1)


In [9]:
# 5. 종가 정규화 : 신경망이 학습하기 쉽도록 데이터의 크기 범위를 조정
print('정규화 전')
print(data[:5])

# 최솟값은 0, 최댓값은 1로 변환
scaler = MinMaxScaler()
data = scaler.fit_transform(data)
print('정규화 후')
print(data[:5])

정규화 전
[[65024.5078125 ]
 [65119.57421875]
 [64644.24609375]
 [63408.39453125]
 [63978.796875  ]]
정규화 후
[[0.0531093 ]
 [0.05341231]
 [0.05189727]
 [0.04795816]
 [0.04977624]]


In [10]:
# 6. RNN 학습 데이터 생성
# 과거 20일 종가로 다음 날 종가 예측
X = []
y = []

# 반복문 코드 설명
# X[0]과 y[0] 입력값은
# 이전 셀 display(df.head(21))결과에서 
# 2023-10-19~2023-11-15 20개 데이터가 X[0]에
# 2023-11-16일 종가가 y[0]에 입력된다.
# X[1]에는 2023-10-20 ~ 2023-11-16
# y[1]에는 2023-11-17일 종가가 입력 ... 
for i in range(0, len(data) - 20): # 남은 20개는 그 다음날 종가를 없기에 학습에 사용할 수 없다. (전체 데이터 수 - 20개)의 데이터가 만들어 진다 
    X.append(data[i:i+20]) # i부터 ㅑ+20은 배열이다. X에 배열이 추가됨. X[i]번째 = [[값],[값],[값],...]
    y.append(data[i+20, 0]) # i+20행의 0번째 열은 값이다. y에값이 추가됨. y[i]번째 = y[값]

X = np.array(X)
y = np.array(y)

# X: (샘플 수, 20일, 특성 1개)
# y: (샘플 수,)
display(X.shape)
display(y.shape)

print(X[0])
print(y[0])

(709, 20, 1)

(709,)

[[0.0531093 ]
 [0.05341231]
 [0.05189727]
 [0.04795816]
 [0.04977624]
 [0.04977624]
 [0.0485642 ]
 [0.05371533]
 [0.05704842]
 [0.05674537]
 [0.06068448]
 [0.06068448]
 [0.05765439]
 [0.05886641]
 [0.05947243]
 [0.05916943]
 [0.06038148]
 [0.06462359]
 [0.06644164]
 [0.06553254]]
0.06613858636640144


In [11]:
# 7. 학습 데이터와 테스트 데이터 분리
# 시간 순서 유지: 앞 80% 학습, 뒤 20% 평가
from sklearn.model_selection import train_test_split

# train_test_split을 이용한 데이터 분할 (시간 순서 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    shuffle=False  # 필수: 무작위 셔플 방지
)
print(X_train.shape)
print(X_test.shape)

(567, 20, 1)
(142, 20, 1)


In [12]:
# 8. SimpleRNN 모델 생성
model = Sequential([
    SimpleRNN(50, input_shape=(20, 1)),
    Dense(1)
])

# 손실 함수와 최적화 알고리즘 설정
model.compile(
    optimizer="adam",
    loss="mse"
)

d:\data_work\machine_learning\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [13]:
# 9. 모델 학습
model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    shuffle=False
)

Epoch 1/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0012    
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0024 
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0015     
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.4540e-04 
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.4616e-04 
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.3835e-04 
Epoch 7/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.4594e-04 
Epoch 8/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.3077e-04 
Epoch 9/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 8.5607e-04 
Epoch 10/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0010     
Epoch 11/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0011     
Epoch 12/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.8930e-04 
Epoch 13/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.9580e-04 
Epoch 14/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.7444e-04 
Epoch 15/20
18/18 ━━

In [14]:
# 10. 테스트 데이터 예측
# RNN모델의 predict 입력값은 3차원 배열, 반환값(예측값)은 2차원 배열 이다
pred = model.predict(X_test)

# MinMaxScaler()정규화된 값을 원래 주가 단위로 복원
pred = scaler.inverse_transform(pred) # 예측종가

#MinMaxScaler.inverse_transform 메서드 입력값은 2차원 배열이다.reshape로 y_test를 1차원에서 2차원으로 변경
# inverse_transform의 반환값 역시 2차원 배열
actual = scaler.inverse_transform(y_test.reshape(-1, 1)) 

# 실제 종가와 예측 종가 비교를 위해 데이터 프레임 생성 후 출력창에 직접 확인
result = pd.DataFrame({
    "실제 종가": actual.flatten(), # 데이터 프레임의 열값으로 넣기 위해 1차원으로 변경
    "예측 종가": pred.flatten() # 데이터 프레임의 열값으로 넣기 위해 1차원으로 변경
})

display(result)

# 결정계수로 확인 : 1에 가까울 수록 설명력이 좋은 모델
r2 = r2_score(actual, pred)
print(f"R² Score (결정계수): {r2:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


,실제 종가,예측 종가
0,188401.828125,190517.015625
1,179530.015625,193977.156250
2,179530.015625,184991.312500
3,176105.781250,184602.578125
4,167015.812500,177897.734375
...,...,...
137,215500.000000,217366.375000
138,218000.000000,209067.265625
139,211500.000000,208350.375000
140,223500.000000,205801.687500


R² Score (결정계수): 0.7965


In [15]:
# 11. 다음 거래일 종가 예측
# 가장 최근 20일 종가를 입력으로 사용하여 다음날의 주가 예측

last_20_days = data[-20:].reshape(1, 20, 1)

next_price = model.predict(last_20_days)

# 원래 주가 단위로 복원
next_price = scaler.inverse_transform(next_price)

display(pd.DataFrame({
    "예측 대상": ["다음 거래일"],
    "예측 종가": next_price.flatten()
}))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


,예측 대상,예측 종가
0,다음 거래일,219725.984375
